In [2]:
import sys
sys.path.append("..")

In [3]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [4]:
def get_model_adv_pga(X_0, X_r, cfr, alpha, lamb, pga_max_iter: int = 100):
    X_0 = torch.tensor(np.stack(X_0)).float()
    X_r = torch.tensor(np.stack(X_r)).float()
    
    loss_fn = torch.nn.BCELoss(reduction='mean')
    cfr_adv = deepcopy(cfr)
    optimizer = optim.Adam(cfr_adv.parameters(), maximize=True)
    weights_min = [cfr.fc1.weight.data-alpha, cfr.fc2.weight.data-alpha, cfr.fc3.weight.data-alpha, cfr.out.weight.data-alpha]
    weights_max = [cfr.fc1.weight.data+alpha, cfr.fc2.weight.data+alpha, cfr.fc3.weight.data+alpha, cfr.out.weight.data+alpha]
    bias_min = [cfr.fc1.bias.data-alpha, cfr.fc2.bias.data-alpha, cfr.fc3.bias.data-alpha, cfr.out.bias.data-alpha]
    bias_max = [cfr.fc1.bias.data+alpha, cfr.fc2.bias.data+alpha, cfr.fc3.bias.data+alpha, cfr.out.bias.data+alpha]
        
    loss = torch.tensor(1.)
    loss_diff = 1
    i = 0
    # while loss_diff > 1e-4:
    for epoch in range(pga_max_iter):
        prev_loss = loss.clone().detach()
        optimizer.zero_grad()
        
        f_x = cfr_adv(X_r)
        y_target = torch.ones(f_x.shape).float()
        bce_loss = loss_fn(f_x, y_target)
        cost = torch.dist(X_r, X_0, 1)
        loss = bce_loss + lamb*cost
        
        loss.backward()
        optimizer.step()
        
        loss_diff = torch.dist(prev_loss, loss, 1)
        i += 1
        
        # clamp model parameters to -alpha, alpha range
        cfr_adv.fc1.weight.data = cfr_adv.fc1.weight.data.clamp(weights_min[0], weights_max[0])
        cfr_adv.fc2.weight.data = cfr_adv.fc2.weight.data.clamp(weights_min[1], weights_max[1])
        cfr_adv.fc3.weight.data = cfr_adv.fc3.weight.data.clamp(weights_min[2], weights_max[2])
        cfr_adv.out.weight.data = cfr_adv.out.weight.data.clamp(weights_min[3], weights_max[3])
        
        cfr_adv.fc1.bias.data = cfr_adv.fc1.bias.data.clamp(bias_min[0], bias_max[0])
        cfr_adv.fc2.bias.data = cfr_adv.fc2.bias.data.clamp(bias_min[1], bias_max[1])
        cfr_adv.fc3.bias.data = cfr_adv.fc3.bias.data.clamp(bias_min[2], bias_max[2])
        cfr_adv.out.bias.data = cfr_adv.out.bias.data.clamp(bias_min[3], bias_max[3])
    
    wnorms = [
        torch.dist(cfr.fc1.weight.data, cfr_adv.fc1.weight.data, torch.inf),
        torch.dist(cfr.fc2.weight.data, cfr_adv.fc2.weight.data, torch.inf),
        torch.dist(cfr.fc3.weight.data, cfr_adv.fc3.weight.data, torch.inf),
        torch.dist(cfr.out.weight.data, cfr_adv.out.weight.data, torch.inf),
    ]
    
    bnorms = [
        torch.dist(cfr.fc1.bias.data, cfr_adv.fc1.bias.data, torch.inf),
        torch.dist(cfr.fc2.bias.data, cfr_adv.fc2.bias.data, torch.inf),
        torch.dist(cfr.fc3.bias.data, cfr_adv.fc3.bias.data, torch.inf),
        torch.dist(cfr.out.bias.data, cfr_adv.out.bias.data, torch.inf),
    ]
    
    # print(f'Final Loss: {loss}')
    # print(f'Num Iterations: {i}')
    # print(f'weights_alpha, bias_alpha: {max(wnorms), max(bnorms)}')
            
    return cfr_adv

# LInf

In [247]:
data = pd.read_pickle("../results/recourse/nn_german_alg1_0.5_0.1_0.pkl")
idx = 20
X_0 = torch.tensor(np.stack(data['x_0']), dtype=torch.float32)
X_r = torch.tensor(np.stack(data['x_r']), dtype=torch.float32)
theta_0 = data['theta_0']
alpha = data['alpha'].unique()[0]
lamb = data['lambda'].unique()[0]

In [248]:
cfr = NN(X_0.shape[1])
cfr.model.load_state_dict(torch.load("../results/recourse_model/german_0.pth"))

<All keys matched successfully>

In [249]:
# cfr.model[0].weight.data = cfr.model[0].weight.data.to(torch.float64)
# cfr.model[2].weight.data = cfr.model[2].weight.data.to(torch.float64)
# cfr.model[4].weight.data = cfr.model[4].weight.data.to(torch.float64)
# cfr.model[6].weight.data = cfr.model[6].weight.data.to(torch.float64)

# cfr.model.parameters()

In [250]:
weights_min = [cfr.model[0].weight.data-alpha, cfr.model[2].weight.data-alpha, cfr.model[4].weight.data-alpha, cfr.model[6].weight.data-alpha]
weights_max = [cfr.model[0].weight.data+alpha, cfr.model[2].weight.data+alpha, cfr.model[4].weight.data+alpha, cfr.model[6].weight.data+alpha]
bias_min = [cfr.model[0].bias.data-alpha, cfr.model[2].bias.data-alpha, cfr.model[4].bias.data-alpha, cfr.model[6].bias.data-alpha]
bias_max = [cfr.model[0].bias.data+alpha, cfr.model[2].bias.data+alpha, cfr.model[4].bias.data+alpha, cfr.model[6].bias.data+alpha]
cfr_adv = deepcopy(cfr.model)
optimizer = optim.Adam(cfr_adv.parameters(), maximize=True)
# optimizer = optim.SGD(cfr_adv.parameters(), lr=0.1, maximize=True)

loss_fn = torch.nn.BCELoss(reduction='mean')

In [251]:
cfr_adv

Sequential(
  (0): Linear(in_features=7, out_features=50, bias=True)
  (1): ReLU()
  (2): Linear(in_features=50, out_features=100, bias=True)
  (3): ReLU()
  (4): Linear(in_features=100, out_features=200, bias=True)
  (5): ReLU()
  (6): Linear(in_features=200, out_features=1, bias=True)
  (7): Sigmoid()
)

In [252]:
pga_max_iter = 150
loss = torch.tensor(1.)
loss_diff = 1

loss_tracker = []

# i = 0
# while loss_diff > 1e-4:
for epoch in range(pga_max_iter):
    loss_tracker.append(loss.clone().detach())

    # prev_loss = loss.clone().detach()
    optimizer.zero_grad()
    
    f_x = cfr_adv(X_r)
    y_target = torch.ones(f_x.shape).to(torch.float32)
    bce_loss = loss_fn(f_x, y_target)
    cost = torch.dist(X_r, X_0, 1)
    loss = bce_loss + lamb*cost
    
    loss.backward()
    optimizer.step()
    
    # loss_diff = torch.dist(prev_loss, loss, 1)
    # i += 1
    
    # clamp model parameters to -alpha, alpha range
    cfr_adv[0].weight.data = cfr_adv[0].weight.data.clamp(weights_min[0], weights_max[0])
    cfr_adv[2].weight.data = cfr_adv[2].weight.data.clamp(weights_min[1], weights_max[1])
    cfr_adv[4].weight.data = cfr_adv[4].weight.data.clamp(weights_min[2], weights_max[2])
    cfr_adv[6].weight.data = cfr_adv[6].weight.data.clamp(weights_min[3], weights_max[3])
    
    cfr_adv[0].bias.data = cfr_adv[0].bias.data.clamp(bias_min[0], bias_max[0])
    cfr_adv[2].bias.data = cfr_adv[2].bias.data.clamp(bias_min[1], bias_max[1])
    cfr_adv[4].bias.data = cfr_adv[4].bias.data.clamp(bias_min[2], bias_max[2])
    cfr_adv[6].bias.data = cfr_adv[6].bias.data.clamp(bias_min[3], bias_max[3])

In [ ]:
# INstace-wise
cfr_adv(X_r).detach()

tensor([4.5182e-22])

In [ ]:
# Population-wise
cfr_adv(X_r[idx]).detach()

tensor([2.2665e-21])

In [246]:
px.scatter(loss_tracker)

# L1

In [266]:
data = pd.read_pickle("../results/recourse/nn_german_alg1_0.1_0.1_0.pkl")
ind = 14
X_0 = torch.tensor(np.stack(data['x_0']), dtype=torch.float32)[ind]
X_r = torch.tensor(np.stack(data['x_r']), dtype=torch.float32)[ind]
theta_0 = data['theta_0']
alpha = data['alpha'].unique()[0]
lamb = data['lambda'].unique()[0]

In [267]:
cfr = NN(X_0.shape[0])
cfr.model.load_state_dict(torch.load("../results/recourse_model/german_0.pth"))

<All keys matched successfully>

In [268]:
cfr_adv = deepcopy(cfr.model)
# optimizer = optim.Adam(cfr_adv.parameters(), maximize=True)
optimizer = optim.SGD(cfr_adv.parameters(), lr=0.1, maximize=True)
loss_fn = torch.nn.BCELoss(reduction='mean')

In [269]:
def l1_projection(x, epsilon):
        """
        Projects a tensor x onto the L1-ball of radius epsilon.
        """
        if torch.norm(x, p=1) <= epsilon:
            return x

        # Sort absolute values in descending order
        abs_x = torch.abs(x)
        sorted_abs_x, _ = torch.sort(abs_x, descending=True)

        # Find the threshold tau
        cumsum = torch.cumsum(sorted_abs_x, dim=0)
        num_elements = torch.arange(1, len(sorted_abs_x) + 1, device=x.device)
        tau_candidates = (cumsum - epsilon) / num_elements
        
        # Select the largest tau such that sorted_abs_x[i] > tau
        valid_taus = tau_candidates[sorted_abs_x > tau_candidates]
        if len(valid_taus) == 0:
            tau = 0.0 # Should not happen if norm > epsilon
        else:
            tau = valid_taus[-1]

        # Apply the projection
        return torch.sign(x) * torch.relu(abs_x - tau)

In [270]:
def project_l1_ball(x: torch.Tensor, eps: float) -> torch.Tensor:
    """
    Project x onto the L1-ball {z : ||z||_1 <= eps}.
    Works for any shape. Preserves device/dtype.
    """
    # Flatten
    orig_shape = x.shape
    v = x.detach().reshape(-1)

    # Already feasible
    if torch.linalg.norm(v, ord=1) <= eps:
        return x

    # Sort |v| descending
    u, _ = torch.sort(v.abs(), descending=True)
    sv = torch.cumsum(u, dim=0)

    j = torch.arange(1, u.numel() + 1, device=v.device, dtype=v.dtype)
    # Find rho = max { j : u_j > (sv_j - eps)/j }
    cond = u * j > (sv - eps)
    rho = torch.nonzero(cond, as_tuple=False).max()
    theta = (sv[rho] - eps) / (rho.to(v.dtype) + 1)

    # Soft-threshold with theta and reshape back
    w = torch.sign(v) * torch.clamp(v.abs() - theta, min=0)
    return w.view(orig_shape)

In [271]:
pga_max_iter = 1000
loss = torch.tensor(1.)
# loss_diff = 1
# i = 0

loss_tracker = []

# while loss_diff > 1e-4:
for epoch in range(pga_max_iter):
    loss_tracker.append(loss.clone().detach())

    # prev_loss = loss.clone().detach()
    optimizer.zero_grad()
    
    f_x = cfr_adv(X_r)
    y_target = torch.ones(f_x.shape).to(torch.float32)
    bce_loss = loss_fn(f_x, y_target)
    cost = torch.dist(X_r, X_0, 1)
    loss = bce_loss + lamb*cost
    
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        for param_adv, param in zip(cfr_adv.parameters(), cfr.model.parameters()):
            delta = param_adv - param
            delta_proj = project_l1_ball(delta, alpha)
            param_adv.copy_(param + delta_proj)

    # # if epoch % 10 == 0:
    # diff_w = [cfr_adv[0].weight.data - cfr.model[0].weight.data,
    #           cfr_adv[2].weight.data - cfr.model[2].weight.data,
    #           cfr_adv[4].weight.data - cfr.model[4].weight.data,
    #           cfr_adv[6].weight.data - cfr.model[6].weight.data,
    #           cfr_adv[0].bias.data - cfr.model[0].bias.data,
    #           cfr_adv[2].bias.data - cfr.model[2].bias.data,
    #           cfr_adv[4].bias.data - cfr.model[4].bias.data,
    #           cfr_adv[6].bias.data - cfr.model[6].bias.data] 
    # diff_w_abs = [torch.abs(w) for w in diff_w]
    # diff_w_abs_flat = torch.cat([w.view(-1) for w in diff_w_abs])
    # divider = torch.sum(diff_w_abs_flat)
    # diff_w_divided = [(w/divider) * alpha for w in diff_w]

    # if divider > alpha:
    #   for w_i, l_i in zip(range(0,8,2), range(0,4,1)):
    #     cfr_adv[w_i].weight.data = cfr.model[w_i].weight.data + diff_w_divided[l_i]
    #     cfr_adv[w_i].bias.data = cfr.model[w_i].bias.data + diff_w_divided[l_i + 4]
    
    # loss_diff = torch.dist(prev_loss, loss, 1)
    # i += 1
    
    

In [272]:
for param in cfr.model.parameters():
    print(param)

Parameter containing:
tensor([[-0.0600,  0.1631, -0.3535, -0.3082, -0.1759,  0.1347,  0.0366],
        [ 0.2834, -0.0700,  0.0946, -0.0445, -0.0457, -0.4378, -0.3050],
        [-0.2295, -0.0032,  0.1139,  0.1697, -0.2494, -0.2224,  0.1724],
        [ 0.3773, -0.1070,  0.2593, -0.0702,  0.1288,  0.3009, -0.3672],
        [-0.2819, -0.1090, -0.1187,  0.3500, -0.2642, -0.1701, -0.2213],
        [-0.3870, -0.1957,  0.3660,  0.1159,  0.1974,  0.0485, -0.2134],
        [-0.0139, -0.4027, -0.2408, -0.2765,  0.2379,  0.2525, -0.1378],
        [-0.0845,  0.2463,  0.4258,  0.2607,  0.1240,  0.1984, -0.1898],
        [ 0.0066, -0.3498, -0.1935, -0.1953,  0.1556,  0.1243, -0.2237],
        [ 0.0908,  0.2015, -0.0449, -0.0223,  0.1213,  0.2915,  0.3709],
        [-0.2964, -0.1106,  0.1893,  0.2722,  0.3223,  0.2922,  0.0767],
        [-0.3679,  0.1378, -0.1921, -0.3522,  0.3444,  0.2770, -0.3770],
        [ 0.1391, -0.0553, -0.1135, -0.1730,  0.1814, -0.1868,  0.1228],
        [ 0.2800,  0.3209,  0

In [273]:
diff_w = [cfr_adv[0].weight.data - cfr.model[0].weight.data,
              cfr_adv[2].weight.data - cfr.model[2].weight.data,
              cfr_adv[4].weight.data - cfr.model[4].weight.data,
              cfr_adv[6].weight.data - cfr.model[6].weight.data,
              cfr_adv[0].bias.data - cfr.model[0].bias.data,
              cfr_adv[2].bias.data - cfr.model[2].bias.data,
              cfr_adv[4].bias.data - cfr.model[4].bias.data,
              cfr_adv[6].bias.data - cfr.model[6].bias.data] 

diff_w_abs = [torch.abs(w) for w in diff_w]
diff_w_abs_flat = torch.cat([w.view(-1) for w in diff_w_abs])
diff_w_abs_flat.sum()

tensor(0.8000)

In [274]:
diff_w

[tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.00

In [275]:
px.scatter(loss_tracker)

In [265]:
#Population
cfr_adv(X_r[ind])

tensor([0.3818], grad_fn=<SigmoidBackward0>)

In [276]:
# Instance
cfr_adv(X_r)

tensor([0.2813], grad_fn=<SigmoidBackward0>)

# Test L1

In [108]:
diff_w = [cfr_adv[0].weight.data - cfr.model[0].weight.data,
          cfr_adv[2].weight.data - cfr.model[2].weight.data,
          cfr_adv[4].weight.data - cfr.model[4].weight.data,
          cfr_adv[6].weight.data - cfr.model[6].weight.data,
          cfr_adv[0].bias.data - cfr.model[0].bias.data,
          cfr_adv[2].bias.data - cfr.model[2].bias.data,
          cfr_adv[4].bias.data - cfr.model[4].bias.data,
          cfr_adv[6].bias.data - cfr.model[6].bias.data] 
diff_w_abs = [torch.abs(w) for w in diff_w]
diff_w_abs_flat = torch.cat([w.view(-1) for w in diff_w_abs])
divider = torch.sum(diff_w_abs_flat)
diff_w_divided = [(w/divider) * alpha for w in diff_w]

In [109]:
divider

tensor(0.)

In [167]:
# Make sure the sum returns alpha value back
diff_w_divided_flat = torch.cat([w.view(-1) for w in diff_w_divided])
alpha_tmp = torch.sum(torch.abs(diff_w_divided_flat))
alpha_tmp

tensor(0.1000)

In [168]:
for w_i, l_i in zip(range(0,8,2), range(0,4,1)):
    cfr_adv[w_i].weight.data = cfr.model[w_i].weight.data + diff_w_divided[l_i]
    cfr_adv[w_i].bias.data = cfr.model[w_i].bias.data + diff_w_divided[l_i + 4]

# Instace Wise L-Inf

In [12]:
data = pd.read_pickle("../results/recourse/nn_german_alg1_0.5_0.1_0.pkl")
X_0 = torch.tensor(np.stack(data['x_0']), dtype=torch.float32)
X_r = torch.tensor(np.stack(data['x_r']), dtype=torch.float32)
theta_0 = data['theta_0']
alpha = data['alpha'].unique()[0]
lamb = data['lambda'].unique()[0]

In [13]:
cfr = NN(X_0.shape[1])
cfr.model.load_state_dict(torch.load("../results/recourse_model/german_0.pth"))

<All keys matched successfully>

In [ ]:
ind = 0
x_0 = X_0[ind]
x_r = X_r[ind]
alpha_sign = torch.sign(x_r) * -alpha

In [37]:
print(x_r)
print(alpha_sign)

tensor([-0.0749,  1.1331,  2.1508,  1.0000,  0.0000,  0.0000,  0.0000])
tensor([ 0.1000, -0.1000, -0.1000, -0.1000, -0.0000, -0.0000, -0.0000])


In [35]:
cfr.model[0].weight.data

tensor([[-0.0600,  0.1631, -0.3535, -0.3082, -0.1759,  0.1347,  0.0366],
        [ 0.2834, -0.0700,  0.0946, -0.0445, -0.0457, -0.4378, -0.3050],
        [-0.2295, -0.0032,  0.1139,  0.1697, -0.2494, -0.2224,  0.1724],
        [ 0.3773, -0.1070,  0.2593, -0.0702,  0.1288,  0.3009, -0.3672],
        [-0.2819, -0.1090, -0.1187,  0.3500, -0.2642, -0.1701, -0.2213],
        [-0.3870, -0.1957,  0.3660,  0.1159,  0.1974,  0.0485, -0.2134],
        [-0.0139, -0.4027, -0.2408, -0.2765,  0.2379,  0.2525, -0.1378],
        [-0.0845,  0.2463,  0.4258,  0.2607,  0.1240,  0.1984, -0.1898],
        [ 0.0066, -0.3498, -0.1935, -0.1953,  0.1556,  0.1243, -0.2237],
        [ 0.0908,  0.2015, -0.0449, -0.0223,  0.1213,  0.2915,  0.3709],
        [-0.2964, -0.1106,  0.1893,  0.2722,  0.3223,  0.2922,  0.0767],
        [-0.3679,  0.1378, -0.1921, -0.3522,  0.3444,  0.2770, -0.3770],
        [ 0.1391, -0.0553, -0.1135, -0.1730,  0.1814, -0.1868,  0.1228],
        [ 0.2800,  0.3209,  0.1424, -0.3053, -0.225

In [ ]:
layer1_adv_w = cfr.model[0].weight.data + alpha_sign
layer1_adv

In [42]:
layer1_adv_bias = cfr.model[0].bias.data - alpha

In [ ]:
import torch
import torch.nn as nn
from copy import deepcopy
from typing import Union, Dict, List

def weight_space_pgd_single_instance(
    model: nn.Module,
    x: torch.Tensor,
    y_target: Union[None, torch.Tensor] = None,
    steps: int = 20,
    step_size: float = 1e-3,
    alpha: Union[float, Dict[str, float]] = 1e-2,
    lamb: float = 0.0,
    device: Union[None, torch.device] = None,
):
    """
    Perform PGD (L_inf) in weight space to maximize loss on a single input x.
    - model: original model (expects model(x) -> sigmoid output since your NN uses Sigmoid())
    - x: single instance tensor, shape [D] or [1, D]
    - y_target: if None we use ones (i.e., push output towards 1); can pass a label tensor shape [1,1] or [1]
    - steps, step_size: PGD steps and per-step size
    - alpha: scalar or dict for per-param type (e.g. {'weight': 1e-2, 'bias': 1e-3}) — used as bound around original params
    - lamb: optional extra cost term (keeps same as your loop)
    Returns: (model_adv, final_loss)
    """
    # device and dtypes
    device = device or next(model.parameters()).device
    model = model.to(device)
    model.eval()

    # Ensure x shape is [1, D]
    x = x.to(device)
    if x.ndim == 1:
        x = x.unsqueeze(0)

    # If user did not provide a target, default to ones (matches your earlier usage)
    # Ensure y_target shape is compatible with model output (sigmoid -> shape [1,1])
    if y_target is None:
        # one-hot style for BCEloss with Sigmoid output
        # model returns shape [1,1], so y_target should be [1,1]
        y_target = torch.ones((x.shape[0], 1), dtype=torch.float32, device=device)
    else:
        y_target = y_target.to(device)

    # Make a deepcopy so original model is unchanged
    model_adv = deepcopy(model).to(device)
    model_adv.eval()

    # Collect the parameters we actually want to attack (Linear layers weights & biases)
    attacked_params: List[torch.nn.Parameter] = []
    # Also snapshot originals in same order
    orig_params: List[torch.Tensor] = []

    # assuming the network is nn.Sequential with Linear layers at indices 0,2,4,6
    # more robust: iterate modules and take Linear layers
    for m in model_adv.modules():
        if isinstance(m, nn.Linear):
            if m.weight is not None:
                attacked_params.append(m.weight)
                orig_params.append(m.weight.detach().clone())
            if m.bias is not None:
                attacked_params.append(m.bias)
                orig_params.append(m.bias.detach().clone())

    # Helper to get alpha for a param (bias vs weight)
    def _alpha_for_param(param, default_alpha):
        if isinstance(default_alpha, dict):
            # user provided mapping for 'weight' and 'bias'
            # detect by shape: bias is 1-D
            return float(default_alpha.get('bias' if param.ndim == 1 else 'weight', list(default_alpha.values())[0]))
        return float(default_alpha)

    # Loss function (use BCELoss because your model ends with Sigmoid)
    loss_fn = nn.BCELoss(reduction='mean')

    # PGD loop (manual ascent)
    for step in range(steps):
        # zero grads
        for p in attacked_params:
            if p.grad is not None:
                p.grad.zero_()

        # forward
        logits = model_adv(x)               # already has Sigmoid in your model
        bce_loss = loss_fn(logits, y_target)

        # optional extra cost term (if you are optimizing X_r too; keep for parity)
        # For single-instance weight-space attack, cost term usually not included, but left here for compatibility
        # If you do not have X_0 in this scope, just omit lamb*cost (set lamb=0 when calling)
        total_loss = bce_loss  # + lamb * cost if you compute input cost

        # ascend: we want to maximize total_loss. Compute grads of total_loss w.r.t. weights
        total_loss.backward()

        # manual step: sign-step for Linf (FGSM-style step in weight space)
        with torch.no_grad():
            for p, p0 in zip(attacked_params, orig_params):
                if p.grad is None:
                    continue
                # ascend
                p.add_(step_size * torch.sign(p.grad))
                # project (L_inf clamp around original)
                a = _alpha_for_param(p, alpha)
                p.clamp_(p0 - a, p0 + a)

    # final loss on the single instance
    with torch.no_grad():
        final_loss = loss_fn(model_adv(x), y_target).item()

    return model_adv, final_loss
